# Exercise 2: Cell Image Segmentation with U-Net CNN

**Lecture 10: Neural Networks and Deep Learning**

## Learning Objectives
1. Build U-Net architecture for semantic segmentation
2. Handle image data with PyTorch
3. Understand encoder-decoder architectures
4. Evaluate segmentation with IoU/Dice metrics

## Biological Context
Automatic cell segmentation in microscopy images for quantitative analysis

In [ ]:
!pip install -q torch torchvision scikit-image matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, segmentation, draw
from skimage.transform import resize

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device}')

## Part 1: Generate Synthetic Cell Images

In [ ]:
def generate_cell_image(size=256, n_cells=10):
    '''Generate synthetic microscopy image with cells'''
    image = np.zeros((size, size))
    mask = np.zeros((size, size), dtype=np.uint8)
    
    for i in range(n_cells):
        center_x = np.random.randint(20, size-20)
        center_y = np.random.randint(20, size-20)
        radius = np.random.randint(10, 25)
        rr, cc = draw.disk((center_y, center_x), radius, shape=(size, size))
        image[rr, cc] = 0.7 + np.random.rand() * 0.3
        mask[rr, cc] = 1
    
    # Add noise
    image += np.random.randn(size, size) * 0.1
    image = np.clip(image, 0, 1)
    
    return image.astype(np.float32), mask.astype(np.float32)

# Generate dataset
print('Generating 500 training samples...')
X_train, y_train = [], []
for _ in range(500):
    img, mask = generate_cell_image()
    X_train.append(img)
    y_train.append(mask)

X_val, y_val = [], []
for _ in range(50):
    img, mask = generate_cell_image()
    X_val.append(img)
    y_val.append(mask)

print(f'Train: {len(X_train)}, Val: {len(X_val)}')

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i in range(3):
    axes[0, i].imshow(X_train[i], cmap='gray')
    axes[0, i].set_title(f'Image {i+1}')
    axes[0, i].axis('off')
    axes[1, i].imshow(y_train[i], cmap='gray')
    axes[1, i].set_title(f'Mask {i+1}')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

## Part 2: Build U-Net Architecture

**U-Net** (Ronneberger et al., 2015):
- Encoder (contracting path): Convolutions + pooling
- Decoder (expanding path): Upsampling + convolutions
- Skip connections: Concatenate encoder features to decoder

In [ ]:
class UNet(nn.Module):
    '''U-Net for cell segmentation'''
    
    def __init__(self):
        super().__init__()
        
        # Encoder
        self.enc1 = self.conv_block(1, 64)
        self.enc2 = self.conv_block(64, 128)
        self.enc3 = self.conv_block(128, 256)
        self.enc4 = self.conv_block(256, 512)
        self.pool = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = self.conv_block(512, 1024)
        
        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = self.conv_block(1024, 512)  # 1024 = 512 + 512 (skip)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = self.conv_block(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = self.conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = self.conv_block(128, 64)
        
        # Output
        self.out = nn.Conv2d(64, 1, 1)
        self.sigmoid = nn.Sigmoid()
    
    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        
        # Bottleneck
        b = self.bottleneck(self.pool(e4))
        
        # Decoder with skip connections
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)
        
        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)
        
        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        
        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        
        return self.sigmoid(self.out(d1))

model = UNet().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Part 3: Training

**Loss**: Dice Loss (better for imbalanced segmentation)

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        pred = pred.view(-1)
        target = target.view(-1)
        intersection = (pred * target).sum()
        dice = (2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)
        return 1 - dice

# Training setup
criterion = DiceLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Convert to tensors
X_train_t = torch.FloatTensor(X_train).unsqueeze(1)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_val_t = torch.FloatTensor(X_val).unsqueeze(1)
y_val_t = torch.FloatTensor(y_val).unsqueeze(1)

train_dataset = torch.utils.data.TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

print('Training...')
for epoch in range(20):
    model.train()
    total_loss = 0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}')

print('Training complete!')

## Part 4: Evaluation & Visualization

In [ ]:
# Evaluate on validation set
model.eval()
with torch.no_grad():
    sample_imgs = X_val_t[:3].to(device)
    sample_masks = y_val_t[:3].to(device)
    predictions = model(sample_imgs)

# Visualize
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i in range(3):
    axes[i, 0].imshow(sample_imgs[i, 0].cpu(), cmap='gray')
    axes[i, 0].set_title('Input')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(sample_masks[i, 0].cpu(), cmap='gray')
    axes[i, 1].set_title('Ground Truth')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(predictions[i, 0].cpu(), cmap='gray')
    axes[i, 2].set_title('Prediction')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

print('\nKey Takeaways:')
print('1. U-Net uses skip connections to preserve spatial information')
print('2. Encoder extracts features, decoder reconstructs segmentation')
print('3. Dice loss handles class imbalance better than BCE')
print('4. Applications: cell counting, morphology analysis, tracking')